<a href="https://colab.research.google.com/github/GyaanFlow/face-expression-app/blob/main/Notebook/face_expression_v4_pooled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Face expression v4 - pooled data, honest CV, pushed to the live app

The linear part trains on the 78 Py-Feat feature numbers. The optional photo teacher fine-tunes pretrained backbones on the photos in your zips and Google Drive folder (GPU recommended), and its scores are distilled into the linear model.

What it does
1. collects every features.csv it can find: the Expression Pool export zip, assignment zips, and your Google Drive folder (downloaded automatically from the shared link, resumable)
2. removes duplicate people with a tolerance match (the same person shows up in many zips, sometimes rounded differently), and checks that copies carry the same labels
3. compares linear models with person-grouped cross validation, plus a nested CV number that does not flatter the winner
4. prints all metrics, logs them to Weights and Biases (optional)
5. saves model.json for the grader and pushes config.json + model.safetensors to the Hugging Face repo the Streamlit app reads

About 100 percent: it is not reachable. The grader can only use a linear map of the 156 inputs, and two expression pairs (sad / hopelessness, scepticism / uncertainty) look almost the same in these numbers. With the 66 unique people in the pool the honest accuracy on unseen people is about 0.71. The training accuracy printed later is higher, but it is not what the grader will see.

Photos are read from the zips into memory only for the teacher; they are never uploaded to Hugging Face or Weights and Biases. The course asks you not to put classmates' photos on outside services, and Colab is one, so set `use_photos=False` if that applies to you.

## Settings

In [1]:
import os, sys, glob, json, time, zipfile, hashlib, warnings, subprocess
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")

CFG = dict(
    hf_repo="23f1000805/face-expression-v3",   # the repo the live app reads, keep the same name
    push_to_hf=True,
    use_wandb=True,
    wandb_project="face-expression-v4",
    run_name="face_expr_v4_pooled",
    # where to look for data (globs). zips are searched for features.csv / pooled-features.csv only
    zip_globs=["/content/*.zip", "/kaggle/input/**/*.zip", "./*.zip", "./data/**/*.zip"],
    csv_globs=["/content/*.csv", "/kaggle/input/**/*.csv", "./data/**/*.csv"],
    use_drive=True,
    # option 1: the folder as a shortcut in My Drive (mounted, no file limit). If the path does not exist ...
    drive_dir="/content/drive/MyDrive/dataset_P1_face_expressoin",
    # option 2: ... the shared link is downloaded automatically (folder must be "Anyone with the link")
    drive_folder_url="https://drive.google.com/drive/folders/1zE1bgLxoQSFzKQ8N9B8BQRB9Py3lAR2W",
    # photo teacher (your fine-tuning recipe): needs a GPU, trained only on people that have all 10 photos
    use_photos=True,
    photo_models=[
        dict(name="convnext_small", id="convnext_small.in12k_ft_in1k", size=224, img_arg=False, epochs=12, bs=32, lr=3e-4, ld=0.9, dp=0.2),
        dict(name="clip_base", id="vit_base_patch16_clip_224.openai", size=224, img_arg=False, epochs=12, bs=32, lr=2e-5, ld=0.75, dp=0.1),
    ],
    pretrained=True,
    crop_box=dict(cx=0.5, cy=0.38, side=0.78),
    crop_px=256,
    teacher_folds=5,
    min_photo_people=20,
    guard_frac=0.6, guard_loss=2.28, guard_retries=2,
    student_grid=[(0.5, 2.0), (0.7, 2.0), (0.5, 1.0)],   # (alpha, temperature)
    min_gain=0.01,
    upload_teacher=False,    # fold weights are large (hundreds of MB), the app does not need them
    folds=5,
    repeats=10,
    nested_repeats=3,
    ridge_alphas=[300.0, 1000.0, 3000.0],
    lr_Cs=[0.005, 0.01, 0.02],
    lda_shrinks=[0.5],
    use_listwise=True,       # slower, about 20 s per candidate per repeat
    seed=0,
)
if os.environ.get("SMOKE"):
    CFG.update(repeats=2, nested_repeats=1, push_to_hf=False, use_wandb=False, use_listwise=False, use_drive=False,
               pretrained=False, crop_px=96, teacher_folds=3, min_photo_people=10, student_grid=[(0.5, 2.0)],
               photo_models=[dict(name="resnet18_smoke", id="resnet18", size=96, img_arg=False, epochs=2, bs=32, lr=1e-3, ld=None, dp=0.0)])

OUT = "face_expr_v4_run"
os.makedirs(OUT + "/reports", exist_ok=True)
print("colab:", IN_COLAB, "| kaggle:", IN_KAGGLE)

colab: True | kaggle: True


## Install and imports

In [2]:
need = ["numpy", "pandas", "scipy", "sklearn", "safetensors", "huggingface_hub", "matplotlib"]
missing = []
for m in need:
    try:
        __import__(m)
    except ImportError:
        missing.append("scikit-learn" if m == "sklearn" else m)
if CFG["use_wandb"]:
    try:
        import wandb
    except ImportError:
        missing.append("wandb")
if CFG["use_photos"]:
    for m in ["torch", "timm", "torchvision", "PIL"]:
        try:
            __import__(m)
        except ImportError:
            missing.append("pillow" if m == "PIL" else m)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=False)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

LABELS = ["neutral", "happy", "sad", "angry", "surprised", "contempt",
          "restrained-amusement", "scepticism", "uncertainty", "hopelessness"]
T0 = time.time()
LOG = []
def log(msg):
    line = f"[{(time.time() - T0) / 60:6.1f} min] {msg}"
    print(line)
    LOG.append(line)

## Tokens (Hugging Face and Weights and Biases)

In [3]:
def get_secret(name):
    v = os.environ.get(name)
    if v:
        return v
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return None

HF_TOKEN = get_secret("HF_TOKEN")
WB_KEY = get_secret("WANDB_API_KEY")
if CFG["push_to_hf"] and not HF_TOKEN:
    log("no HF_TOKEN found, the push to Hugging Face will be skipped")
    CFG["push_to_hf"] = False

run = None
if CFG["use_wandb"] and WB_KEY:
    import wandb
    wandb.login(key=WB_KEY)
    run = wandb.init(project=CFG["wandb_project"], name=CFG["run_name"], config=CFG)
    log(f"wandb run {run.url}")
else:
    log("wandb off")

def wb(d):
    if run is not None:
        run.log(d)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f1000805 (23f1000805-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[   0.1 min] wandb run https://wandb.ai/23f1000805-indian-institute-of-technology-madras/face-expression-v4/runs/i4jni1j6


## Collect data
Optional Google Drive: in Colab run `from google.colab import drive; drive.mount('/content/drive')`, add the shared folder to My Drive, and set `drive_dir`. A public folder link in `drive_folder_url` is downloaded with gdown instead (gdown stops at 50 files per folder).

In [4]:
def download_drive_folder(url, out):
    """Download every file of a public Drive folder. Lists first, then fetches file by file with retries,
    skipping files already on disk, so a rerun only fetches what is missing."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown>=5"], check=False)
    import gdown
    os.makedirs(out, exist_ok=True)
    try:
        items = gdown.download_folder(url=url, output=out, quiet=True, skip_download=True, use_cookies=False)
    except Exception as e:
        log(f"could not list the Drive folder ({str(e)[:150]}). Is it shared as 'Anyone with the link'?")
        return
    log(f"drive folder lists {len(items)} files")
    if len(items) >= 50:
        log("WARNING: gdown lists at most 50 files per folder. Split bigger folders or use the mounted Drive path.")
    got, failed = 0, []
    for it in items:
        dest = it.local_path
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            got += 1
            continue
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        ok = False
        for attempt in range(3):
            try:
                gdown.download(id=it.id, output=dest, quiet=True, resume=True, use_cookies=False)
                ok = os.path.exists(dest) and os.path.getsize(dest) > 0
                if ok:
                    break
            except Exception as e:
                time.sleep(3 * (attempt + 1))
        if ok:
            got += 1
        else:
            failed.append(it.path)
    log(f"drive download: {got} files ready, {len(failed)} failed")
    if failed:
        log("failed files (Drive download quota is the usual cause, rerun later): " + ", ".join(failed[:10]))

# 1) mounted Drive path, if present (works for any number of files)
if IN_COLAB and CFG["use_drive"] and CFG["drive_dir"].startswith("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        log(f"drive mount skipped: {str(e)[:100]}")
# 2) otherwise download the shared folder link
if CFG["use_drive"] and not os.path.isdir(CFG["drive_dir"]) and CFG["drive_folder_url"]:
    log(f"{CFG['drive_dir']} not found, downloading the shared folder instead")
    CFG["drive_dir"] = "drive_download"
    download_drive_folder(CFG["drive_folder_url"], CFG["drive_dir"])

zip_paths, csv_paths = set(), set()
for pat in CFG["zip_globs"]:
    zip_paths.update(glob.glob(pat, recursive=True))
for pat in CFG["csv_globs"]:
    csv_paths.update(glob.glob(pat, recursive=True))
if CFG["use_drive"] and os.path.isdir(CFG["drive_dir"]):
    zip_paths.update(glob.glob(CFG["drive_dir"] + "/**/*.zip", recursive=True))
    csv_paths.update(glob.glob(CFG["drive_dir"] + "/**/*.csv", recursive=True))
log(f"found {len(zip_paths)} zip files and {len(csv_paths)} csv files")

tables = []   # (source name, dataframe, {photo_id: (zip path, inner name)})

def add_table(name, df, photos):
    if "intended_expression" not in df.columns:
        return
    if "person_id" not in df.columns:
        if "photo_id" not in df.columns:
            return
        df["person_id"] = df["photo_id"].astype(str).str.rsplit("-", n=1).str[0]
    tables.append((name, df, photos))

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")
for zp in sorted(zip_paths):
    try:
        z = zipfile.ZipFile(zp)
        names = z.namelist()
    except Exception:
        log(f"not a readable zip: {zp}")
        continue
    for inner in names:
        base = inner.split("/")[-1].lower()
        if not base.endswith(".csv") or not ("feature" in base or "/uploads/" in "/" + inner):
            continue
        folder = inner[: len(inner) - len(inner.split("/")[-1])]
        photos = {}
        if CFG["use_photos"]:
            for n in names:
                if n.startswith(folder + "photos/") and n.lower().endswith(IMG_EXT):
                    photos[os.path.splitext(n.split("/")[-1])[0]] = (zp, n)
        try:
            with z.open(inner) as f:
                add_table(f"{os.path.basename(zp)}:{inner}", pd.read_csv(f, dtype={"person_id": str, "photo_id": str}), photos)
        except Exception as e:
            log(f"could not read {inner} in {zp}: {e}")
for cp in sorted(csv_paths):
    photos = {}
    pdir = os.path.join(os.path.dirname(cp), "photos")
    if CFG["use_photos"] and os.path.isdir(pdir):
        for n in os.listdir(pdir):
            if n.lower().endswith(IMG_EXT):
                photos[os.path.splitext(n)[0]] = (None, os.path.join(pdir, n))
    try:
        add_table(cp, pd.read_csv(cp, dtype={"person_id": str, "photo_id": str}), photos)
    except Exception as e:
        log(f"could not read {cp}: {e}")

if not tables:
    raise SystemExit("no features.csv found. Upload the Expression Pool zip or assignment zips next to the notebook.")
log(f"tables with photos: {sum(1 for t in tables if t[2])}")

COLS = [c for c in tables[0][1].columns if c.startswith(("bs_", "au_", "emotion_"))]
log(f"{len(tables)} tables, {len(COLS)} feature columns")
assert len(COLS) == 78, "expected 78 feature columns"

[   0.2 min] drive mount skipped: Error: credential propagation was unsuccessful
[   0.2 min] /content/drive/MyDrive/dataset_P1_face_expressoin not found, downloading the shared folder instead
[   0.3 min] drive folder lists 13 files
[   0.9 min] drive download: 13 files ready, 0 failed
[   0.9 min] found 15 zip files and 0 csv files
[   1.0 min] tables with photos: 13
[   1.0 min] 243 tables, 78 feature columns


## Remove duplicate people
The same person appears in many students' zips, and the pool export stores numbers rounded to 6 significant digits, so exact hashing misses copies (that mistake inflated a test score from 0.71 to 0.80). Here two photos count as the same when every feature agrees within a small tolerance, and two people are the same when 5 or more of their photos match. Only the first copy is kept, and the labels of the copies are compared.

In [5]:
ATOL, RTOL = 1e-5, 1e-4

kept = []
bank = np.zeros((0, len(COLS)))      # every kept photo row
bank_owner = np.zeros(0, dtype=int)  # which kept person the row belongs to
bank_lab = []
bank_photo = []
listed, bad, dup, conflicts = 0, 0, 0, 0
for name, df, photos in tables:
    if any(c not in df.columns for c in COLS):
        log(f"skip {name}: missing feature columns")
        continue
    df = df.copy()
    df["intended_expression"] = df["intended_expression"].astype(str).str.strip().str.lower()
    df[COLS] = df[COLS].apply(pd.to_numeric, errors="coerce")
    for pid in list(dict.fromkeys(df["person_id"].astype(str))):
        g = df[df["person_id"].astype(str) == pid]
        listed += 1
        x = g[COLS].to_numpy(float)
        if len(g) != 10 or sorted(g["intended_expression"]) != sorted(LABELS) or not np.isfinite(x).all():
            bad += 1
            continue
        refs = [photos.get(str(ph)) if "photo_id" in g.columns else None for ph in g.get("photo_id", [None] * 10)]
        owner = None
        if len(bank):
            # rows of the bank that match each new row within tolerance
            close = (np.abs(x[:, None, :] - bank[None, :, :]) <= ATOL + RTOL * np.abs(bank[None, :, :])).all(-1)
            hits = {}
            for i in range(10):
                for o in set(bank_owner[close[i]]):
                    hits[o] = hits.get(o, 0) + 1
            best = max(hits.items(), key=lambda t: t[1]) if hits else (None, 0)
            if best[1] >= 5:
                owner = best[0]
                for i in range(10):
                    js = np.where(close[i] & (bank_owner == owner))[0]
                    if len(js) and bank_lab[js[0]] != g["intended_expression"].iloc[i]:
                        conflicts += 1
        if owner is not None:
            dup += 1
            # a copy may bring the photos the first copy did not have
            for i in range(10):
                js = np.where(close[i] & (bank_owner == owner))[0]
                if len(js) and bank_photo[js[0]] is None and refs[i] is not None:
                    bank_photo[js[0]] = refs[i]
            continue
        bank_photo += refs
        bank = np.vstack([bank, x])
        bank_owner = np.concatenate([bank_owner, np.full(10, len(kept))])
        bank_lab += list(g["intended_expression"])
        kept.append(dict(src=name, pid=pid, df=g.reset_index(drop=True)))

log(f"people listed {listed}, broken {bad}, duplicates {dup}, label conflicts {conflicts}, unique {len(kept)}")
if conflicts:
    log("WARNING: some copies of the same photo carry different labels, check the sources")
if len(kept) < 10:
    raise SystemExit("fewer than 10 unique people, add more data")

def make_inputs(g):
    raw = g[COLS].to_numpy(float)
    rk = g[COLS].rank(method="average").to_numpy()
    return np.hstack([raw, (rk - 1) / 9])

def center(g):
    raw = g[COLS].to_numpy(float)
    return raw - raw.mean(axis=0)

X = np.vstack([make_inputs(p["df"]) for p in kept])
XC = np.hstack([np.vstack([center(p["df"]) for p in kept]), X[:, len(COLS):]])
y = np.concatenate([p["df"]["intended_expression"].map(LABELS.index).to_numpy() for p in kept])
G = np.repeat(np.arange(len(kept)), 10)
N_PEOPLE = len(kept)
PHOTO = bank_photo                  # same row order as X
full = [p for p in range(N_PEOPLE) if all(PHOTO[p * 10 + i] is not None for i in range(10))]
log(f"people with all 10 photos: {len(full)} of {N_PEOPLE}")
print(X.shape, "people:", N_PEOPLE)
wb(dict(people_listed=listed, people_unique=N_PEOPLE, duplicates=dup))

[   2.9 min] people listed 7362, broken 0, duplicates 7296, label conflicts 0, unique 66
[   2.9 min] people with all 10 photos: 66 of 66
(660, 156) people: 66


## Models and the matching score

In [6]:
def match_pred(S, g):
    pred = np.zeros(len(g), dtype=int)
    for p in np.unique(g):
        k = np.where(g == p)[0]
        r, c = linear_sum_assignment(-S[k])
        pred[k[r]] = c
    return pred

def match_acc(S, yy, g):
    return float((match_pred(S, g) == yy).mean())

def fit_one(kind, a, Xt, yt):
    sc = StandardScaler().fit(Xt)
    Z = sc.transform(Xt)
    if kind == "ridge":
        W = Ridge(alpha=a).fit(Z, np.eye(10)[yt] - 0.1).coef_
    elif kind == "lr":
        W = LogisticRegression(C=a, max_iter=4000).fit(Z, yt).coef_
    else:
        W = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=a).fit(Z, yt).coef_
    return W / sc.scale_      # folds the scaler in, the grader multiplies raw inputs

def fit_listwise(Xt, yt, gt, lam=0.3, mix=0.8):
    # softmax over expressions plus softmax over the person's photos for each expression
    # (a Plackett-Luce style loss that looks like the one-to-one matching the grader does)
    from scipy.optimize import minimize
    from scipy.special import logsumexp
    sc = StandardScaler().fit(Xt)
    Z = sc.transform(Xt)
    n, f = Z.shape
    Y = np.eye(10)[yt]
    groups = [np.where(gt == q)[0] for q in np.unique(gt)]
    def fun(w):
        Wm = w.reshape(10, f)
        S = Z @ Wm.T
        lr_ = S - logsumexp(S, axis=1, keepdims=True)
        loss = -(Y * lr_).sum() * (1 - mix)
        Gr = (1 - mix) * (np.exp(lr_) - Y)
        for k in groups:
            lc = S[k] - logsumexp(S[k], axis=0, keepdims=True)
            loss += -(Y[k] * lc).sum() * mix
            Gr[k] += mix * (np.exp(lc) - Y[k])
        return loss / n + lam * (Wm ** 2).sum() / 2, ((Gr.T @ Z) / n + lam * Wm).ravel()
    r = minimize(fun, np.zeros(10 * f), jac=True, method="L-BFGS-B", options={"maxiter": 400})
    return r.x.reshape(10, f) / sc.scale_

def build(spec, idx):
    # spec: list of (kind, param, use_centred). parts are averaged after scaling each to unit score spread.
    # centred training is allowed: a per person offset adds the same amount to every assignment, so it cancels.
    W = np.zeros((10, X.shape[1]))
    for kind, a, ctr in spec:
        Xt = XC[idx] if ctr else X[idx]
        if kind == "list":
            Wi = fit_listwise(Xt, y[idx], G[idx], lam=a)
        else:
            Wi = fit_one(kind, a, Xt, y[idx])
        W += Wi / (np.std(X[idx] @ Wi.T) + 1e-12)
    return W

SPECS = {}
for a in CFG["ridge_alphas"]:
    SPECS[f"ridge_{a:g}"] = [("ridge", a, False)]
for c in CFG["lr_Cs"]:
    SPECS[f"lr_{c:g}"] = [("lr", c, False)]
for sh in CFG["lda_shrinks"]:
    SPECS[f"lda_{sh:g}"] = [("lda", sh, False)]
SPECS["lr_0.01_centred"] = [("lr", 0.01, True)]
SPECS["ens_ridge1000_lr0.01"] = [("ridge", 1000.0, False), ("lr", 0.01, False)]
SPECS["ens_ridge1000_lr0.01_lda0.5"] = [("ridge", 1000.0, False), ("lr", 0.01, False), ("lda", 0.5, False)]
if CFG["use_listwise"]:
    SPECS["listwise_0.3"] = [("list", 0.3, False)]
    SPECS["ens_ridge_lr_listwise"] = [("ridge", 1000.0, False), ("lr", 0.01, False), ("list", 0.3, False)]
print(len(SPECS), "candidates:", list(SPECS))

def folds_for(g, seed, k):
    ps = np.unique(g)
    rng = np.random.RandomState(seed)
    rng.shuffle(ps)
    m = {}
    for i in range(len(ps)):
        m[ps[i]] = i % k
    return np.array([m[p] for p in g])

def cv(spec, idx, repeats, seed0=0):
    # idx: indices of the rows to use (all rows, or an outer training fold)
    ga, ya = G[idx], y[idx]
    accs, oof = [], None
    for r in range(repeats):
        f = folds_for(ga, seed0 + r, CFG["folds"])
        S = np.zeros((len(idx), 10))
        for k in range(CFG["folds"]):
            tr, te = idx[f != k], f == k
            S[te] = X[idx[te]] @ build(spec, tr).T
        accs.append(match_acc(S, ya, ga))
        if oof is None:
            oof = S
    return np.array(accs), oof

12 candidates: ['ridge_300', 'ridge_1000', 'ridge_3000', 'lr_0.005', 'lr_0.01', 'lr_0.02', 'lda_0.5', 'lr_0.01_centred', 'ens_ridge1000_lr0.01', 'ens_ridge1000_lr0.01_lda0.5', 'listwise_0.3', 'ens_ridge_lr_listwise']


## Photo teacher (your fine-tuning recipe, optional)
Same recipe as the teacher notebook: pretrained timm backbone with a new 10 way head, AdamW with layer-wise lr decay, warmup then cosine, label smoothing 0.1, flip test time augmentation, and the collapse guard that restarts a fold with a smaller learning rate when the loss stays at chance. Folds are split by person, so every photo gets a score from a model that never saw that person. Photos are read straight from the zips, cropped and kept in memory; nothing is uploaded.

Research note: on small face sets, fine-tuned backbones usually beat frozen linear probes, but with about 60 people the teacher in your earlier runs stayed near 0.5 to 0.6 and the student gained nothing. So the teacher only counts if the student beats the linear model on held-out people by `min_gain`.

In [7]:
TL = None          # teacher log-probabilities, one row per photo (NaN where no photo)
TEACHER_ACC = None
if CFG["use_photos"] and len(full) >= CFG["min_photo_people"]:
    import io, gc
    import torch, timm
    from PIL import Image, ImageOps
    from torchvision.transforms import v2
    from timm.optim import create_optimizer_v2
    from scipy.special import logsumexp
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    AMP = DEVICE == "cuda"
    log(f"teacher on {DEVICE}, {len(full)} people")
    if DEVICE == "cpu" and not os.environ.get("SMOKE"):
        log("WARNING: no GPU. Fine-tuning on CPU takes hours. In Colab: Runtime -> Change runtime type -> T4 GPU.")

    open_zips = {}
    def read_photo(ref):
        zp, n = ref
        if zp is None:
            return Image.open(n)
        if zp not in open_zips:
            open_zips[zp] = zipfile.ZipFile(zp)
        return Image.open(io.BytesIO(open_zips[zp].read(n)))

    def crop(ref, box, px):
        img = ImageOps.exif_transpose(read_photo(ref)).convert("RGB")
        w, h = img.size
        side = box["side"] * w
        x0 = min(max(box["cx"] * w - side / 2, 0), max(w - side, 0))
        y0 = min(max(box["cy"] * h - side / 2, 0), max(h - side, 0))
        return np.asarray(img.crop((int(x0), int(y0), int(x0 + side), int(y0 + side))).resize((px, px), Image.BICUBIC))

    rows_t = np.concatenate([np.arange(p * 10, p * 10 + 10) for p in full])
    crops, bad_rows = [], 0
    for r in rows_t:
        try:
            crops.append(crop(PHOTO[r], CFG["crop_box"], CFG["crop_px"]))
        except Exception:
            bad_rows += 1
            crops.append(np.zeros((CFG["crop_px"], CFG["crop_px"], 3), dtype=np.uint8))
    if bad_rows:
        log(f"{bad_rows} photos could not be read and were replaced by blanks")
    CROPS_T = torch.from_numpy(np.stack(crops)).permute(0, 3, 1, 2).contiguous()
    yt_all = y[rows_t]
    gt_all = G[rows_t]
    log(f"crops {tuple(CROPS_T.shape)}")

    def build_model(spec):
        kw = {}
        if spec.get("img_arg"):
            kw["img_size"] = spec["size"]
        if spec.get("dp"):
            kw["drop_path_rate"] = spec["dp"]
        net = timm.create_model(spec["id"], pretrained=CFG["pretrained"], num_classes=10, **kw)
        dc = timm.data.resolve_data_config({}, model=net)
        return net, [float(v) for v in dc["mean"]], [float(v) for v in dc["std"]]

    def make_opt(net, spec, lr):
        try:
            return create_optimizer_v2(net, opt="adamw", lr=lr, weight_decay=0.05, layer_decay=spec.get("ld"))
        except Exception as e:
            log(f"layer decay not available for {spec['id']}: {str(e)[:80]}")
            return torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.05)

    def make_aug(size):
        return v2.Compose([v2.RandomResizedCrop(size, scale=(0.75, 1.0), ratio=(0.9, 1.1), antialias=True),
                           v2.RandomHorizontalFlip(), v2.RandomRotation(8), v2.ColorJitter(0.3, 0.3, 0.2, 0.02)])

    @torch.no_grad()
    def predict(net, idx, spec, mean_t, std_t):
        net.eval()
        rs = v2.Resize((spec["size"], spec["size"]), antialias=True)
        out = []
        for a in range(0, len(idx), 64):
            xb = torch.stack([rs(CROPS_T[i]) for i in idx[a:a + 64]]).float().to(DEVICE)
            xb = (xb - mean_t) / std_t
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                lg = (net(xb) + net(xb.flip(-1))) / 2
            out.append(lg.float().cpu())
        return torch.cat(out).numpy()

    def train_fold(spec, tr, te, seed, tag):
        for attempt in range(CFG["guard_retries"] + 1):
            lr = spec["lr"] * (0.4 ** attempt)
            torch.manual_seed(seed + 1000 * attempt)
            rng = np.random.RandomState(seed + 1000 * attempt)
            net, mean, std = build_model(spec)
            net.to(DEVICE)
            mean_t = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1) * 255
            std_t = torch.tensor(std, device=DEVICE).view(1, 3, 1, 1) * 255
            opt = make_opt(net, spec, lr)
            for gr in opt.param_groups:
                gr["base_lr"] = gr["lr"]            # timm already applied lr_scale for layer decay
            aug = make_aug(spec["size"])
            ytr = torch.from_numpy(yt_all).long()
            steps_per = int(np.ceil(len(tr) / spec["bs"]))
            total, warm = steps_per * spec["epochs"], steps_per
            scaler = torch.amp.GradScaler("cuda", enabled=AMP)
            guard_ep = max(1, int(np.ceil(CFG["guard_frac"] * spec["epochs"])))
            step, collapsed, ep_loss = 0, False, 0.0
            for ep in range(spec["epochs"]):
                net.train()
                perm = rng.permutation(tr)
                tot = 0.0
                for a in range(0, len(perm), spec["bs"]):
                    b = perm[a:a + spec["bs"]]
                    if len(b) < 2:
                        continue
                    f = (step + 1) / warm if step < warm else 0.5 * (1 + np.cos(np.pi * (step - warm) / max(1, total - warm)))
                    for gr in opt.param_groups:
                        gr["lr"] = gr["base_lr"] * f
                    xb = torch.stack([aug(CROPS_T[i]) for i in b]).float().to(DEVICE)
                    xb = (xb - mean_t) / std_t
                    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                        loss = torch.nn.functional.cross_entropy(net(xb), ytr[b].to(DEVICE), label_smoothing=0.1)
                    opt.zero_grad(set_to_none=True)
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                    tot += float(loss) * len(b)
                    step += 1
                ep_loss = tot / len(perm)
                wb({f"teacher_loss/{tag}": ep_loss})
                if ep + 1 == guard_ep and ep_loss > CFG["guard_loss"]:
                    collapsed = True
                    break
            log(f"  {tag}: last loss {ep_loss:.3f}, lr {lr:.1e}" + (" (stuck at chance)" if collapsed else ""))
            if collapsed and attempt < CFG["guard_retries"]:
                del net, opt
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()
                continue
            break
        lg = predict(net, te, spec, mean_t, std_t)
        sd = {k: v.detach().cpu().half().contiguous().clone() for k, v in net.state_dict().items()} if CFG["upload_teacher"] else None
        del net, opt
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        return lg, sd

    TEACH = {}
    for si, spec in enumerate(CFG["photo_models"]):
        t0 = time.time()
        try:
            f = folds_for(gt_all, 7 + si, CFG["teacher_folds"])
            oof = np.zeros((len(rows_t), 10), dtype=np.float32)
            for k in range(CFG["teacher_folds"]):
                tr, te = np.where(f != k)[0], np.where(f == k)[0]
                lg, sd = train_fold(spec, tr, te, 100 * si + k, f"{spec['name']}/fold{k + 1}")
                oof[te] = lg
                if sd is not None:
                    from safetensors.torch import save_file as save_t
                    os.makedirs(f"{OUT}/teacher/{spec['name']}", exist_ok=True)
                    save_t(sd, f"{OUT}/teacher/{spec['name']}/fold{k}.safetensors")
            lp = oof - logsumexp(oof, axis=1, keepdims=True)
            acc_t = match_acc(lp, yt_all, gt_all)
            TEACH[spec["name"]] = lp
            log(f"teacher {spec['name']}: matching accuracy on unseen people {acc_t:.4f} ({(time.time() - t0) / 60:.1f} min)")
            wb({f"teacher/{spec['name']}": acc_t})
        except Exception as e:
            log(f"teacher {spec['name']} failed: {type(e).__name__}: {str(e)[:150]}")
    if TEACH:
        lp = np.mean(list(TEACH.values()), axis=0)
        TEACHER_ACC = match_acc(lp, yt_all, gt_all)
        TL = np.full((len(y), 10), np.nan)
        TL[rows_t] = lp
        log(f"teacher ensemble {list(TEACH)}: {TEACHER_ACC:.4f} on {len(full)} people")
        # same people, linear model, for a fair comparison
        sub_idx = rows_t
        a_lin, _ = cv(SPECS["ens_ridge1000_lr0.01"], sub_idx, 3, 50)
        log(f"linear model on the same people: {a_lin.mean():.4f}")
elif CFG["use_photos"]:
    log(f"photo teacher skipped: only {len(full)} people have all 10 photos (need {CFG['min_photo_people']})")
else:
    log("photo teacher off")

[   2.9 min] teacher on cuda, 66 people
[   3.0 min] crops (660, 3, 256, 256)


model.safetensors: reconstructing file:   0%|          |  0.00B /  201MB            

model.safetensors: downloading bytes:           |  0.00B            

[   4.6 min]   convnext_small/fold1: last loss 2.325, lr 3.0e-04 (stuck at chance)
[   6.0 min]   convnext_small/fold1: last loss 2.307, lr 1.2e-04 (stuck at chance)
[   7.6 min]   convnext_small/fold1: last loss 2.311, lr 4.8e-05 (stuck at chance)
[   9.1 min]   convnext_small/fold2: last loss 2.318, lr 3.0e-04 (stuck at chance)
[  10.5 min]   convnext_small/fold2: last loss 2.317, lr 1.2e-04 (stuck at chance)
[  12.7 min]   convnext_small/fold2: last loss 1.679, lr 4.8e-05
[  14.1 min]   convnext_small/fold3: last loss 2.340, lr 3.0e-04 (stuck at chance)
[  15.6 min]   convnext_small/fold3: last loss 2.325, lr 1.2e-04 (stuck at chance)
[  17.7 min]   convnext_small/fold3: last loss 1.783, lr 4.8e-05
[  19.2 min]   convnext_small/fold4: last loss 2.324, lr 3.0e-04 (stuck at chance)
[  20.6 min]   convnext_small/fold4: last loss 2.320, lr 1.2e-04 (stuck at chance)
[  22.1 min]   convnext_small/fold4: last loss 2.314, lr 4.8e-05 (stuck at chance)
[  23.5 min]   convnext_small/fold5: las

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  599MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[  28.6 min]   clip_base/fold1: last loss 2.312, lr 2.0e-05 (stuck at chance)
[  30.5 min]   clip_base/fold1: last loss 1.889, lr 8.0e-06
[  31.8 min]   clip_base/fold2: last loss 2.312, lr 2.0e-05 (stuck at chance)
[  33.8 min]   clip_base/fold2: last loss 1.852, lr 8.0e-06
[  35.1 min]   clip_base/fold3: last loss 2.304, lr 2.0e-05 (stuck at chance)
[  37.0 min]   clip_base/fold3: last loss 1.924, lr 8.0e-06
[  38.4 min]   clip_base/fold4: last loss 2.313, lr 2.0e-05 (stuck at chance)
[  39.7 min]   clip_base/fold4: last loss 2.307, lr 8.0e-06 (stuck at chance)
[  41.6 min]   clip_base/fold4: last loss 2.184, lr 3.2e-06
[  43.6 min]   clip_base/fold5: last loss 1.689, lr 2.0e-05
[  43.6 min] teacher clip_base: matching accuracy on unseen people 0.5303 (16.4 min)
[  43.6 min] teacher ensemble ['convnext_small', 'clip_base']: 0.5636 on 66 people
[  43.6 min] linear model on the same people: 0.7081


## Leaderboard (repeated person-grouped CV)

In [8]:
ALL = np.arange(len(y))
rows, OOF = [], {}
for name, spec in SPECS.items():
    t = time.time()
    a, oof = cv(spec, ALL, CFG["repeats"], CFG["seed"])
    OOF[name] = oof
    rows.append(dict(name=name, mean=a.mean(), std=a.std(), min=a.min(), max=a.max(), sec=round(time.time() - t, 1)))
    log(f"{name:30s} {a.mean():.4f} +- {a.std():.4f}")
    wb({f"cv/{name}": a.mean()})
LB = pd.DataFrame(rows).sort_values("mean", ascending=False).reset_index(drop=True)
LB.to_csv(OUT + "/reports/leaderboard.csv", index=False)
print(LB.to_string(index=False))
BEST = LB.loc[0, "name"]
log(f"best: {BEST} {LB.loc[0, 'mean']:.4f}")

[  43.6 min] ridge_300                      0.6920 +- 0.0123
[  43.7 min] ridge_1000                     0.7089 +- 0.0120
[  43.7 min] ridge_3000                     0.6942 +- 0.0121
[  43.7 min] lr_0.005                       0.6992 +- 0.0139
[  43.8 min] lr_0.01                        0.7053 +- 0.0125
[  43.9 min] lr_0.02                        0.6973 +- 0.0163
[  44.0 min] lda_0.5                        0.6958 +- 0.0158
[  44.0 min] lr_0.01_centred                0.7059 +- 0.0155
[  44.2 min] ens_ridge1000_lr0.01           0.7121 +- 0.0111
[  44.3 min] ens_ridge1000_lr0.01_lda0.5    0.7108 +- 0.0141
[  44.7 min] listwise_0.3                   0.7023 +- 0.0154
[  45.2 min] ens_ridge_lr_listwise          0.7091 +- 0.0111
                       name     mean      std      min      max  sec
       ens_ridge1000_lr0.01 0.712121 0.011093 0.693939 0.733333  7.9
ens_ridge1000_lr0.01_lda0.5 0.710758 0.014140 0.690909 0.742424  9.4
      ens_ridge_lr_listwise 0.709091 0.011134 0.693939 0.7333

## Student: linear model trained on the teacher's soft scores
Targets are `alpha` times the true label plus `(1 - alpha)` times the teacher softmax at temperature `T`; people without photos keep the hard label. The result is still a 10 x 156 linear model, so it fits the grader. It replaces the linear winner only if it is better by `min_gain` on the same folds.

In [9]:
def fit_student(idx, alpha, temp, C=0.01):
    sc = StandardScaler().fit(X[idx])
    Z = sc.transform(X[idx])
    tgt = np.eye(10)[y[idx]]
    has = ~np.isnan(TL[idx, 0])
    if has.any():
        z = TL[idx][has] / temp
        soft = np.exp(z - z.max(axis=1, keepdims=True))
        soft /= soft.sum(axis=1, keepdims=True)
        tgt[has] = alpha * tgt[has] + (1 - alpha) * soft
    Zr = np.repeat(Z, 10, axis=0)
    yr = np.tile(np.arange(10), len(idx))
    wr = tgt.ravel()
    keep = wr > 1e-4
    m = LogisticRegression(C=C, max_iter=4000).fit(Zr[keep], yr[keep], sample_weight=wr[keep])
    return m.coef_ / sc.scale_

STUDENT = None
if TL is not None:
    base = LB.loc[0, "mean"]
    srows = []
    for alpha, temp in CFG["student_grid"]:
        accs = []
        for r in range(CFG["repeats"]):
            f = folds_for(G, CFG["seed"] + r, CFG["folds"])
            S = np.zeros((len(y), 10))
            for k in range(CFG["folds"]):
                tr = np.where(f != k)[0]
                S[f == k] = X[f == k] @ fit_student(tr, alpha, temp).T
            accs.append(match_acc(S, y, G))
            if r == 0:
                first = S
        nm = f"student_a{alpha}_t{temp}"
        srows.append(dict(name=nm, mean=np.mean(accs), std=np.std(accs), alpha=alpha, temp=temp))
        OOF[nm] = first
        log(f"{nm:30s} {np.mean(accs):.4f} +- {np.std(accs):.4f}")
        wb({f"student/{nm}": np.mean(accs)})
    ST = pd.DataFrame(srows).sort_values("mean", ascending=False).reset_index(drop=True)
    print(ST.to_string(index=False))
    gain = ST.loc[0, "mean"] - base
    if gain >= CFG["min_gain"]:
        STUDENT = dict(alpha=ST.loc[0, "alpha"], temp=ST.loc[0, "temp"])
        BEST = ST.loc[0, "name"]
        LB = pd.concat([ST[["name", "mean", "std"]], LB]).sort_values("mean", ascending=False).reset_index(drop=True)
        log(f"the teacher helps: {BEST} gains {gain:+.4f}, using it")
    else:
        log(f"best student gains {gain:+.4f} (needs {CFG['min_gain']}), keeping {BEST}")
else:
    log("no teacher scores, student skipped")

[  45.6 min] student_a0.5_t2.0              0.7150 +- 0.0089
[  46.1 min] student_a0.7_t2.0              0.7130 +- 0.0136
[  46.6 min] student_a0.5_t1.0              0.7120 +- 0.0085
             name    mean      std  alpha  temp
student_a0.5_t2.0 0.71500 0.008872    0.5   2.0
student_a0.7_t2.0 0.71303 0.013589    0.7   2.0
student_a0.5_t1.0 0.71197 0.008502    0.5   1.0
[  46.6 min] best student gains +0.0029 (needs 0.01), keeping ens_ridge1000_lr0.01


## Nested CV (the number to trust)
Picking the best of many candidates on the same folds is a little optimistic. Here the choice is made inside each outer training fold, so the outer score is a fair estimate of the whole procedure.

In [10]:
nested = []
for r in range(CFG["nested_repeats"]):
    f = folds_for(G, 100 + r, CFG["folds"])
    S = np.zeros((len(y), 10))
    for k in range(CFG["folds"]):
        tr, te = np.where(f != k)[0], f == k
        best_name, best_acc = None, -1
        for name, spec in SPECS.items():
            if spec[0][0] == "list" or any(p[0] == "list" for p in spec):
                continue   # too slow inside the nested loop
            a, _ = cv(spec, tr, 2, 200 + r)
            if a.mean() > best_acc:
                best_name, best_acc = name, a.mean()
        S[te] = X[te] @ build(SPECS[best_name], tr).T
    nested.append(match_acc(S, y, G))
    log(f"nested repeat {r + 1}: {nested[-1]:.4f}")
NESTED = float(np.mean(nested))
log(f"nested CV accuracy on unseen people: {NESTED:.4f} +- {np.std(nested):.4f}")
wb({"nested_cv": NESTED})

[  47.3 min] nested repeat 1: 0.7076
[  48.0 min] nested repeat 2: 0.6939
[  48.7 min] nested repeat 3: 0.6955
[  48.7 min] nested CV accuracy on unseen people: 0.6990 +- 0.0061


## Full metrics of the chosen model

In [11]:
S = OOF[BEST]
pred = match_pred(S, G)
acc = float((pred == y).mean())
per_class = pd.Series([(pred[y == i] == i).mean() for i in range(10)], index=LABELS).round(3)
conf = pd.DataFrame(np.zeros((10, 10), dtype=int), index=["true " + l for l in LABELS], columns=LABELS)
for t, p in zip(y, pred):
    conf.iloc[t, p] += 1
per_person = pd.Series([(pred[G == p] == y[G == p]).mean() for p in range(N_PEOPLE)])

rng = np.random.RandomState(1)
boot = []
for b in range(2000):
    pick = rng.randint(0, N_PEOPLE, N_PEOPLE)
    boot.append(per_person.values[pick].mean())
lo, hi = np.percentile(boot, [2.5, 97.5])

print("matching accuracy (first CV repeat):", round(acc, 4))
print("bootstrap 95% interval over people:", round(lo, 3), "-", round(hi, 3))
print("\nper class accuracy")
print(per_class.to_string())
print("\nconfusion matrix")
print(conf.to_string())
print("\nper person: min", per_person.min(), "median", per_person.median(), "max", per_person.max())

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(conf.values, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(LABELS, rotation=60, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(LABELS)
for i in range(10):
    for j in range(10):
        ax.text(j, i, conf.values[i, j], ha="center", va="center", fontsize=8)
ax.set_title(f"{BEST} - held out people")
fig.tight_layout(); fig.savefig(OUT + "/reports/confusion.png", dpi=110); plt.close(fig)

conf.to_csv(OUT + "/reports/confusion.csv")
per_class.to_csv(OUT + "/reports/per_class.csv")
if run is not None:
    run.log({"confusion": wandb.Image(OUT + "/reports/confusion.png"),
             "per_class": wandb.Table(dataframe=per_class.reset_index().rename(columns={"index": "label", 0: "acc"})),
             "leaderboard": wandb.Table(dataframe=LB)})

matching accuracy (first CV repeat): 0.7333
bootstrap 95% interval over people: 0.683 - 0.782

per class accuracy
neutral                 0.788
happy                   0.879
sad                     0.530
angry                   0.848
surprised               0.955
contempt                0.773
restrained-amusement    0.727
scepticism              0.682
uncertainty             0.591
hopelessness            0.561

confusion matrix
                           neutral  happy  sad  angry  surprised  contempt  restrained-amusement  scepticism  uncertainty  hopelessness
true neutral                    52      0    2      0          0         3                     1           1            6             1
true happy                       0     58    0      0          1         0                     7           0            0             0
true sad                         3      0   35      3          0         2                     1           4            1            17
true angry              

## Learning curve (does more data still help?)

In [12]:
curve = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    accs = []
    for r in range(8):
        rng = np.random.RandomState(300 + r)
        ps = rng.permutation(N_PEOPLE)
        test = ps[: max(2, N_PEOPLE // 5)]
        pool = ps[max(2, N_PEOPLE // 5):]
        train = pool[: max(5, int(len(pool) * frac))]
        tr, te = np.isin(G, train), np.isin(G, test)
        W = fit_student(np.where(tr)[0], STUDENT["alpha"], STUDENT["temp"]) if STUDENT else build(SPECS[BEST], np.where(tr)[0])
        accs.append(match_acc(X[te] @ W.T, y[te], G[te]))
    curve.append((len(train), np.mean(accs), np.std(accs)))
    print(f"{len(train):4d} training people -> {np.mean(accs):.3f} +- {np.std(accs):.3f}")
    wb({"curve/people": len(train), "curve/acc": np.mean(accs)})

  13 training people -> 0.621 +- 0.064
  26 training people -> 0.638 +- 0.070
  39 training people -> 0.664 +- 0.056
  53 training people -> 0.675 +- 0.048


## Final model on all people, grader JSON and checks

In [13]:
W = fit_student(np.arange(len(y)), STUDENT["alpha"], STUDENT["temp"]) if STUDENT else build(SPECS[BEST], np.arange(len(y)))
train_acc = match_acc(X @ W.T, y, G)
log(f"accuracy on the training people (not the real number): {train_acc:.3f}")

grader = {"version": "linear-rank-v1", "weights": [[float(f"{v:.9g}") for v in row] for row in W]}
text = json.dumps(grader, separators=(",", ":"))
open(OUT + "/model.json", "w").write(text)

back = json.loads(text, parse_constant=lambda c: (_ for _ in ()).throw(ValueError(c)))
Wb = np.array(back["weights"], dtype=float)
assert set(back) == {"version", "weights"} and back["version"] == "linear-rank-v1"
assert Wb.shape == (10, 156) and np.isfinite(Wb).all()
assert len(text.encode()) <= 64 * 1024
agree = (match_pred(X @ Wb.T, G) == match_pred(X @ W.T, G)).mean()
log(f"model.json valid, {len(text.encode()) / 1024:.1f} KiB, matching agrees with in-memory weights on {agree * 100:.1f}% of photos")

[  48.8 min] accuracy on the training people (not the real number): 0.833
[  48.8 min] model.json valid, 19.5 KiB, matching agrees with in-memory weights on 100.0% of photos


## Package for the app and push to Hugging Face

In [14]:
from safetensors.numpy import save_file
cv_mean = float(LB.loc[0, "mean"])
config = dict(
    version="linear-rank-v1",
    labels=LABELS,
    feature_columns=COLS,
    rank_mode="minmax",
    input_order="78 raw features then 78 within person ranks",
    chosen=dict(name=BEST, kind="student" if STUDENT else "linear", parts=[] if STUDENT else [[p[0], p[1], bool(p[2])] for p in SPECS[BEST]], used_teacher=bool(STUDENT), teacher_accuracy=TEACHER_ACC),
    cv_accuracy=round(float(LB.loc[0, "mean"]) if STUDENT else NESTED, 4),          # the app shows this number, the nested one is the fair one
    cv_accuracy_best_candidate=round(cv_mean, 4),
    cv_std=round(float(LB.loc[0, "std"]), 4),
    n_people=N_PEOPLE,
    n_photos=int(len(y)),
    run=CFG["run_name"],
    trained=time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()),
)
save_file({"weights": W.astype(np.float32)}, OUT + "/model.safetensors")
json.dump(config, open(OUT + "/config.json", "w"), indent=1)
metrics = dict(teacher_accuracy=TEACHER_ACC, used_teacher=bool(STUDENT), nested_cv=NESTED, best_candidate=BEST, best_candidate_cv=cv_mean, bootstrap_ci95=[lo, hi],
               per_class=per_class.to_dict(), train_accuracy=train_acc, people=N_PEOPLE,
               learning_curve=[dict(people=a, acc=b, std=c) for a, b, c in curve])
json.dump(metrics, open(OUT + "/metrics.json", "w"), indent=1)
open(OUT + "/README.md", "w").write(f"""---
license: mit
tags: [facial-expression, py-feat, linear-model]
---
# Face expression matcher (linear-rank-v1)

Matches ten expressions to the ten photos of one person from 78 Py-Feat features plus their within-person ranks.

- trained on {N_PEOPLE} unique people ({len(y)} photos), duplicates across uploads removed
- model: {BEST}
- nested CV matching accuracy on unseen people: {NESTED:.3f} (chance 0.10)
- weakest pairs: sad vs hopelessness, scepticism vs uncertainty

Files: model.json (grader format), model.safetensors + config.json (used by the Streamlit app), metrics.json, reports/.
Built by [Gaurav Tomar](https://www.linkedin.com/in/gaurav-tomar-630b2a316).
""")
open(OUT + "/run_log.txt", "w").write("\n".join(LOG))

if CFG["push_to_hf"]:
    from huggingface_hub import HfApi, snapshot_download
    api = HfApi(token=HF_TOKEN)
    api.create_repo(CFG["hf_repo"], exist_ok=True, private=True)
    api.upload_folder(folder_path=OUT, repo_id=CFG["hf_repo"],
                      commit_message=f"v4 pooled {N_PEOPLE} people, nested cv {NESTED:.4f}",
                      delete_patterns=["reports/*", "oof/*"] + (["teacher/*"] if CFG["upload_teacher"] else []),
                      ignore_patterns=[] if CFG["upload_teacher"] else ["teacher/*"])
    log(f"pushed to https://huggingface.co/{CFG['hf_repo']}")
    # load back exactly the way the app does and compare
    from safetensors.numpy import load_file
    d = snapshot_download(CFG["hf_repo"], token=HF_TOKEN, allow_patterns=["config.json", "model.safetensors"])
    Wh = load_file(d + "/model.safetensors")["weights"].astype(float)
    ch = json.load(open(d + "/config.json"))
    assert ch["feature_columns"] == COLS and ch["n_people"] == N_PEOPLE
    same = (match_pred(X @ Wh.T, G) == match_pred(X @ W.T, G)).mean()
    log(f"hub check: model loaded back, matching agrees on {same * 100:.1f}% of photos")
    log("now reboot the Streamlit app (menu -> Reboot app) so it loads the new weights")
else:
    log("push skipped")

print("\n========== SUMMARY ==========")
print(f"unique people        : {N_PEOPLE}")
print(f"chosen model         : {BEST}")
print(f"photo teacher        : {TEACHER_ACC if TEACHER_ACC is None else round(TEACHER_ACC, 4)}")
print(f"best candidate CV    : {cv_mean:.4f}")
print(f"nested CV (trust it) : {NESTED:.4f}")
print(f"95% interval         : {lo:.3f} - {hi:.3f}")
print(f"grader file          : {OUT}/model.json")
if run is not None:
    art = wandb.Artifact("face-expression-v4", type="model")
    for f in ["model.json", "model.safetensors", "config.json", "metrics.json", "run_log.txt"]:
        art.add_file(OUT + "/" + f)
    run.log_artifact(art)
    run.summary.update(dict(nested_cv=NESTED, best=BEST, people=N_PEOPLE))
    run.finish()

[  48.8 min] pushed to https://huggingface.co/23f1000805/face-expression-v3


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[  48.9 min] hub check: model loaded back, matching agrees on 100.0% of photos
[  48.9 min] now reboot the Streamlit app (menu -> Reboot app) so it loads the new weights

========== SUMMARY ==========
unique people        : 66
chosen model         : ens_ridge1000_lr0.01
photo teacher        : 0.5636
best candidate CV    : 0.7121
nested CV (trust it) : 0.6990
95% interval         : 0.683 - 0.782
grader file          : face_expr_v4_run/model.json


curve/acc,▁▃▇█
curve/people,▁▃▆█
cv/ens_ridge1000_lr0.01,▁
cv/ens_ridge1000_lr0.01_lda0.5,▁
cv/ens_ridge_lr_listwise,▁
cv/lda_0.5,▁
cv/listwise_0.3,▁
cv/lr_0.005,▁
cv/lr_0.01,▁
cv/lr_0.01_centred,▁
+23,...
